In [1]:
!pip install remotezip
from pathlib import Path
from remotezip import RemoteZip
from tqdm.auto import tqdm
import requests
import zipfile

# --------------------------------------------------
# Configuration
# --------------------------------------------------

OUTPUT_ROOT = Path("/content/RDD2022_China_MotorBike")
COUNTRY_ZIP = Path("/content/RDD2022_China_MotorBike.zip")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Find official Figshare archive
# --------------------------------------------------

FIGSHARE_API = (
    "https://"
    + "api.figshare.com"
    + "/v2/articles/21431547"
)

response = requests.get(FIGSHARE_API, timeout=60)
response.raise_for_status()

figshare_files = response.json()["files"]

outer_archive = next(
    file
    for file in figshare_files
    if file["name"].lower().endswith(".zip")
)

print("Opening official RDD2022 archive...")

# --------------------------------------------------
# Download only the nested China_MotorBike ZIP
# --------------------------------------------------

with RemoteZip(outer_archive["download_url"]) as remote_archive:

    archive_members = remote_archive.namelist()

    country_member = next(
        (
            member
            for member in archive_members
            if "china_motorbike" in member.lower()
            and member.lower().endswith(".zip")
        ),
        None,
    )

    if country_member is None:
        nested_zips = [
            member
            for member in archive_members
            if member.lower().endswith(".zip")
        ]

        raise RuntimeError(
            "China_MotorBike archive was not found.\n"
            f"Nested ZIP files: {nested_zips}"
        )

    member_info = remote_archive.getinfo(country_member)

    print("Found:", country_member)
    print(f"Size : {member_info.file_size / 1024**2:.2f} MB")
    print("Downloading country subset...")

    with remote_archive.open(country_member) as source:
        with open(COUNTRY_ZIP, "wb") as destination:
            with tqdm(
                total=member_info.file_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc="China_MotorBike",
            ) as progress:

                while True:
                    chunk = source.read(1024 * 1024)

                    if not chunk:
                        break

                    destination.write(chunk)
                    progress.update(len(chunk))

# --------------------------------------------------
# Validate and extract
# --------------------------------------------------

if not zipfile.is_zipfile(COUNTRY_ZIP):
    raise RuntimeError("The downloaded country file is not a valid ZIP.")

print("\nExtracting...")

with zipfile.ZipFile(COUNTRY_ZIP, "r") as archive:
    archive.extractall(OUTPUT_ROOT)

# --------------------------------------------------
# Automatically locate the extracted country root
# --------------------------------------------------

country_candidates = [
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_dir()
    and path.name.lower() == "china_motorbike"
    and (path / "train").is_dir()
]

if not country_candidates:
    raise RuntimeError(
        f"Extraction completed, but country root was not found inside {OUTPUT_ROOT}"
    )

COUNTRY_ROOT = country_candidates[0]

TRAIN_IMAGES_DIR = COUNTRY_ROOT / "train/images"
TRAIN_XMLS_DIR = COUNTRY_ROOT / "train/annotations/xmls"
TEST_IMAGES_DIR = COUNTRY_ROOT / "test/images"

# --------------------------------------------------
# Verify
# --------------------------------------------------

train_images = [
    path for path in TRAIN_IMAGES_DIR.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

train_xmls = list(TRAIN_XMLS_DIR.glob("*.xml"))

test_images = [
    path for path in TEST_IMAGES_DIR.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("\nDataset verification")
print("--------------------")
print("Train images :", len(train_images))
print("Train XMLs   :", len(train_xmls))
print("Test images  :", len(test_images))
print("Dataset root :", COUNTRY_ROOT)

assert train_images, "No train images found"
assert train_xmls, "No XML annotations found"
assert test_images, "No test images found"

print("\n✓ Dataset is completely ready.")
print("✓ You can now continue XML → YOLO conversion and sanity checks.")

Opening official RDD2022 archive...
Found: RDD2022/China_MotorBike.zip
Size : 183.13 MB


China_MotorBike:   0%|          | 0.00/183M [00:00<?, ?B/s]


Extracting...

Dataset verification
--------------------
Train images : 1977
Train XMLs   : 1977
Test images  : 500
Dataset root : /content/RDD2022_China_MotorBike/China_MotorBike

✓ Dataset is completely ready.
✓ You can now continue XML → YOLO conversion and sanity checks.


In [2]:
from pathlib import Path
from remotezip import RemoteZip
from tqdm.auto import tqdm
import requests
import zipfile

# --------------------------------------------------
# Configuration
# --------------------------------------------------

OUTPUT_ROOT = Path("/content/RDD2022_China_MotorBike")
COUNTRY_ZIP = Path("/content/RDD2022_China_MotorBike.zip")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Find official Figshare archive
# --------------------------------------------------

FIGSHARE_API = (
    "https://"
    + "api.figshare.com"
    + "/v2/articles/21431547"
)

response = requests.get(FIGSHARE_API, timeout=60)
response.raise_for_status()

figshare_files = response.json()["files"]

outer_archive = next(
    file
    for file in figshare_files
    if file["name"].lower().endswith(".zip")
)

print("Opening official RDD2022 archive...")

# --------------------------------------------------
# Download only the nested China_MotorBike ZIP
# --------------------------------------------------

with RemoteZip(outer_archive["download_url"]) as remote_archive:

    archive_members = remote_archive.namelist()

    country_member = next(
        (
            member
            for member in archive_members
            if "china_motorbike" in member.lower()
            and member.lower().endswith(".zip")
        ),
        None,
    )

    if country_member is None:
        nested_zips = [
            member
            for member in archive_members
            if member.lower().endswith(".zip")
        ]

        raise RuntimeError(
            "China_MotorBike archive was not found.\n"
            f"Nested ZIP files: {nested_zips}"
        )

    member_info = remote_archive.getinfo(country_member)

    print("Found:", country_member)
    print(f"Size : {member_info.file_size / 1024**2:.2f} MB")
    print("Downloading country subset...")

    with remote_archive.open(country_member) as source:
        with open(COUNTRY_ZIP, "wb") as destination:
            with tqdm(
                total=member_info.file_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc="China_MotorBike",
            ) as progress:

                while True:
                    chunk = source.read(1024 * 1024)

                    if not chunk:
                        break

                    destination.write(chunk)
                    progress.update(len(chunk))

# --------------------------------------------------
# Validate and extract
# --------------------------------------------------

if not zipfile.is_zipfile(COUNTRY_ZIP):
    raise RuntimeError("The downloaded country file is not a valid ZIP.")

print("\nExtracting...")

with zipfile.ZipFile(COUNTRY_ZIP, "r") as archive:
    archive.extractall(OUTPUT_ROOT)

# --------------------------------------------------
# Automatically locate the extracted country root
# --------------------------------------------------

country_candidates = [
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_dir()
    and path.name.lower() == "china_motorbike"
    and (path / "train").is_dir()
]

if not country_candidates:
    raise RuntimeError(
        f"Extraction completed, but country root was not found inside {OUTPUT_ROOT}"
    )

COUNTRY_ROOT = country_candidates[0]

TRAIN_IMAGES_DIR = COUNTRY_ROOT / "train/images"
TRAIN_XMLS_DIR = COUNTRY_ROOT / "train/annotations/xmls"
TEST_IMAGES_DIR = COUNTRY_ROOT / "test/images"

# --------------------------------------------------
# Verify
# --------------------------------------------------

train_images = [
    path for path in TRAIN_IMAGES_DIR.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

train_xmls = list(TRAIN_XMLS_DIR.glob("*.xml"))

test_images = [
    path for path in TEST_IMAGES_DIR.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("\nDataset verification")
print("--------------------")
print("Train images :", len(train_images))
print("Train XMLs   :", len(train_xmls))
print("Test images  :", len(test_images))
print("Dataset root :", COUNTRY_ROOT)

assert train_images, "No train images found"
assert train_xmls, "No XML annotations found"
assert test_images, "No test images found"

print("\n✓ Dataset is completely ready.")
print("✓ You can now continue XML → YOLO conversion and sanity checks.")

Opening official RDD2022 archive...
Found: RDD2022/China_MotorBike.zip
Size : 183.13 MB


China_MotorBike:   0%|          | 0.00/183M [00:00<?, ?B/s]


Extracting...

Dataset verification
--------------------
Train images : 1977
Train XMLs   : 1977
Test images  : 500
Dataset root : /content/RDD2022_China_MotorBike/China_MotorBike

✓ Dataset is completely ready.
✓ You can now continue XML → YOLO conversion and sanity checks.


In [3]:
from pathlib import Path
from collections import Counter
import json
import math
import random
import shutil
import xml.etree.ElementTree as ET

import matplotlib.pylab as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image

**configuration**

In [4]:
SEED = 42
CLASS_TO_ID = {
    "D00": 0,
    "D10": 1,
    "D20": 2,
    "D40": 3,
}

CLASS_NAMES = {
    "D00": "Longitudinal crack",
    "D10": "Transverse crack",
    "D20": "Alligator crack",
    "D40": "Pothole",
}
REVIEWED_IGNORED_CLASSES = set()
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png",}

In [5]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


RAW_DATASET_DIR = Path("/content/RDD2022_China_MotorBike/China_MotorBike")

TRAIN_IMAGES_DIR = (RAW_DATASET_DIR / "train" / "images")
TRAIN_XML_DIR = (RAW_DATASET_DIR / "train" / "annotations" / "xmls")
TEST_IMAGES_DIR = (RAW_DATASET_DIR / "test" / "images")


PROCESSED_ALL_DIR = (PROJECT_ROOT / "data" / "processed" / "all")
PROCESSED_IMAGES_DIR = (PROCESSED_ALL_DIR / "images")
PROCESSED_LABELS_DIR = (PROCESSED_ALL_DIR / "labels")


REPORTS_DIR = (PROJECT_ROOT / "reports")
FIGURES_DIR = (REPORTS_DIR / "figures")


print("Project root :", PROJECT_ROOT)
print("Dataset root:", RAW_DATASET_DIR)
print("Train images:", TRAIN_IMAGES_DIR)
print("Train XMLs  :", TRAIN_XML_DIR)
print("Test images :", TEST_IMAGES_DIR)

Project root : /content
Dataset root: /content/RDD2022_China_MotorBike/China_MotorBike
Train images: /content/RDD2022_China_MotorBike/China_MotorBike/train/images
Train XMLs  : /content/RDD2022_China_MotorBike/China_MotorBike/train/annotations/xmls
Test images : /content/RDD2022_China_MotorBike/China_MotorBike/test/images


**File inventory and pairing audit**

In [21]:
def collect_images(directory):
    return sorted(
        path for path in directory.iterdir() if (
            path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        )
    )

image_files = collect_images(TRAIN_IMAGES_DIR)
xml_files = sorted(TRAIN_XML_DIR.glob("*.xml"))
test_image_files = collect_images(TEST_IMAGES_DIR)

image_stems = [path.stem for path in image_files]
xml_stems = [path.stem for path in xml_files]

In [22]:
duplicate_image_stems = [
    stem for stem, count in Counter(image_stems).items() if count > 1
]
duplicate_xml_stems = [
    stem for stem, count in Counter(xml_stems).items() if count > 1
]

images_by_stem = {path.stem: path for path in image_files}
xml_by_stem = {path.stem: path for path in xml_files}

In [23]:
images_without_xml = sorted(set(images_by_stem) - set(xml_by_stem))

xml_without_images = sorted(set(xml_by_stem) - set(images_by_stem))

print("Train images       :", len(image_files))
print("Train XML files    :", len(xml_files))
print("Test images        :", len(test_image_files))
print("Images without XML :", len(images_without_xml))
print("XML without image  :", len(xml_without_images))
print("Duplicate images   :", len(duplicate_image_stems))
print("Duplicate XMLs     :", len(duplicate_xml_stems))

Train images       : 1977
Train XML files    : 1977
Test images        : 500
Images without XML : 0
XML without image  : 0
Duplicate images   : 0
Duplicate XMLs     : 0


**Parse and validate Pascal VOC annotations**

In [30]:
def read_required_number(node, path):
    text = node.findtext(path)
    
    if text is None:
        raise ValueError(f"Missing XML field: {path}")
    
    text = text.strip()
    if not text:
        raise ValueError(f"Empty XML field: {path}")
        
    value = float(text)
    if not math.isfinite(value):
        raise ValueError(f"Non-finite value in field: {path}")

    return value

In [31]:
records = []

audit_errors = []
ignored_class_counts = Counter()
target_class_counts = Counter()
images_per_class = Counter()

box_widths = []
box_heights = []
relative_box_areas = []

very_small_box_count = 0
empty_target_image_count = 0
duplicate_annotation_count = 0

In [32]:
for stem in sorted(image_stems):
    image_path = images_by_stem[stem]
    xml_path = xml_by_stem[stem]
    
    try:
        with Image.open(image_path) as image:
            image.load()
            actual_width, actual_height = image.size
    except Exception as error:
        audit_errors.append(f"{stem}: unreadable image — {error}")
        continue
    
    try:
        root = ET.parse(xml_path).getroot()
    except Exception as error:
        audit_errors.append(f"{stem}: malformed XML — {error}")
        continue
    
    # XML filename check.
    xml_filename = (root.findtext("filename") or "").strip()
    if xml_filename and Path(xml_filename).stem != stem:
        audit_errors.append(
            f"{stem}: XML filename says {xml_filename}"
        )
    
    # XML image-size check.
    try:
        xml_width = int(read_required_number(root, "size/width"))
        xml_height = int(read_required_number(root, "size/height"))
    except Exception as error:
        audit_errors.append(
            f"{stem}: invalid XML size — {error}"
        )
        continue
    
    if xml_width != actual_width or xml_height != actual_height:
        audit_errors.append(
            f"{stem}: XML size {xml_width}x{xml_height}, "
            f"actual size {actual_width}x{actual_height}"
        )

    target_objects = []
    classes_in_this_image = set()
    seen_annotations = set()

    for object_index, object_node in enumerate(root.findall("object")):
        class_code = (object_node.findtext("name") or "").strip()
        if not class_code:
            audit_errors.append(
                f"{stem}: object {object_index} has no class name"
            )
            continue

        if class_code not in CLASS_TO_ID:
            ignored_class_counts[class_code] += 1
            continue

        bbox_node = object_node.find("bndbox")
        if bbox_node is None:
            audit_errors.append(
                f"{stem}: {class_code} has no bndbox"
            )
            continue

        try:
            x1 = read_required_number(bbox_node, "xmin")
            y1 = read_required_number(bbox_node, "ymin")
            x2 = read_required_number(bbox_node, "xmax")
            y2 = read_required_number(bbox_node, "ymax")
        except Exception as error:
            audit_errors.append(
                f"{stem}: invalid box — {error}"
            )
            continue

        if x2 <= x1 or y2 <= y1:
            audit_errors.append(
                f"{stem}: invalid {class_code} box [{x1}, {y1}, {x2}, {y2}]"
            )
            continue

        if x1 < 0 or y1 < 0 or x2 > actual_width or y2 > actual_height:
            audit_errors.append(
                f"{stem}: out-of-bounds {class_code} box [{x1}, {y1}, {x2}, {y2}] "
                f"for image {actual_width}x{actual_height}"
            )
            continue

        annotation_key = (class_code, x1, y1, x2, y2)

        if annotation_key in seen_annotations:
            duplicate_annotation_count += 1
            audit_errors.append(
                f"{stem}: exact duplicate annotation {annotation_key}"
            )
            continue

        seen_annotations.add(annotation_key)

        box_width = x2 - x1
        box_height = y2 - y1

        relative_area = (box_width * box_height / (actual_width * actual_height))

        if box_width < 2 or box_height < 2:
            very_small_box_count += 1

        box_widths.append(box_width)
        box_heights.append(box_height)
        relative_box_areas.append(relative_area)

        target_class_counts[class_code] += 1
        classes_in_this_image.add(class_code)

        target_objects.append({
            "class_code": class_code,
            "class_id": CLASS_TO_ID[class_code],
            "box_xyxy": [x1, y1, x2, y2],
        })

    if not target_objects:
        empty_target_image_count += 1

    images_per_class.update(classes_in_this_image)
    records.append({
        "stem": stem,
        "image_path": image_path,
        "xml_path": xml_path,
        "width": actual_width,
        "height": actual_height,
        "objects": target_objects,
    })

In [33]:
print("Parsed image records :", len(records))
print("Target objects       :", sum(target_class_counts.values()))
print("Empty target images  :", empty_target_image_count)
print("Very small boxes     :", very_small_box_count)
print("Duplicate boxes      :", duplicate_annotation_count)

print("\nTarget class counts:")
for class_code in CLASS_TO_ID:
    print(
        f"{class_code} "
        f"({CLASS_NAMES[class_code]:<20}): "
        f"{target_class_counts[class_code]}"
    )

print("\nNon-target/unknown classes:")
if ignored_class_counts:
    for class_code, count in ignored_class_counts.items():
        print(f"{class_code}: {count}")
else:
    print("None")

Parsed image records : 1977
Target objects       : 4650
Empty target images  : 43
Very small boxes     : 0
Duplicate boxes      : 0

Target class counts:
D00 (Longitudinal crack  ): 2678
D10 (Transverse crack    ): 1096
D20 (Alligator crack     ): 641
D40 (Pothole             ): 235

Non-target/unknown classes:
Repair: 277


In [34]:
if audit_errors:
    print(
        f"Found {len(audit_errors)} critical errors."
    )

    for error in audit_errors[:25]:
        print("-", error)